# подготовка данных

## imports & install

In [ ]:
!pip install -q huggingface_hub pyspark sentence-transformers

In [ ]:
from huggingface_hub import hf_hub_download
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, Window
from pyspark.sql.types import ArrayType, IntegerType, StringType
from pyspark.sql.functions import udf
from sentence_transformers import SentenceTransformer
import json
import numpy as np

## data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/pet_proj_recomendashki'
data_dir = f'{base_path}/data'

spark = SparkSession.builder.appName("k_core").config("spark.driver.memory", "4g").getOrCreate()

Mounted at /content/drive


## reviews

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("McAuley-Lab/Amazon-Reviews-2023", repo_type="dataset")
video_games_files = [f for f in files if "Video_Games" in f]
print('\n'.join(video_games_files))

benchmark/0core/last_out/Video_Games.test.csv
benchmark/0core/last_out/Video_Games.train.csv
benchmark/0core/last_out/Video_Games.valid.csv
benchmark/0core/last_out_w_his/Video_Games.test.csv
benchmark/0core/last_out_w_his/Video_Games.train.csv
benchmark/0core/last_out_w_his/Video_Games.valid.csv
benchmark/0core/rating_only/Video_Games.csv
benchmark/0core/timestamp/Video_Games.test.csv
benchmark/0core/timestamp/Video_Games.train.csv
benchmark/0core/timestamp/Video_Games.valid.csv
benchmark/0core/timestamp_w_his/Video_Games.test.csv
benchmark/0core/timestamp_w_his/Video_Games.train.csv
benchmark/0core/timestamp_w_his/Video_Games.valid.csv
benchmark/5core/last_out/Video_Games.test.csv
benchmark/5core/last_out/Video_Games.train.csv
benchmark/5core/last_out/Video_Games.valid.csv
benchmark/5core/last_out_w_his/Video_Games.test.csv
benchmark/5core/last_out_w_his/Video_Games.train.csv
benchmark/5core/last_out_w_his/Video_Games.valid.csv
benchmark/5core/rating_only/Video_Games.csv
benchmark/5c

In [ ]:
repo_id = "McAuley-Lab/Amazon-Reviews-2023"
filename = "raw/review_categories/Video_Games.jsonl"
hf_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset"
)

In [ ]:
df = spark.read.json(hf_path)
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



## k-core

итеративно убираем пользователей и товары, у которых меньше 5 взаимодействий, пока датасет не перестанет меняться. на каждой итерации план запроса сбрасывается через запись в parquet — иначе он разрастается с каждой итерацией и Spark падает по памяти

In [ ]:
df_k_core = df.select("user_id", "parent_asin")
i = 0
k = 5
new_count = -1
prev_count = -52
while prev_count != new_count:
    prev_count = df_k_core.count()
    user_cnt = df_k_core.groupby("user_id").agg(F.count('*').alias('cnt'))
    valid_users = user_cnt.filter(F.col('cnt') >= k).select("user_id")
    item_cnt = df_k_core.groupby("parent_asin").agg(F.count('*').alias('cnt'))
    valid_items = item_cnt.filter(F.col('cnt') >= k).select("parent_asin")
    df_k_core = df_k_core.join(valid_items, on='parent_asin', how='inner') \
    .join(valid_users, on='user_id', how='inner')

    path = f"/content/kcore_checkpoint_{i}"
    df_k_core.write.mode("overwrite").parquet(path)
    df_k_core = spark.read.parquet(path)

    new_count = df_k_core.count()
    print(i)
    i += 1

0
1
2
3
4
5
6
7
8
9
10
11
12


In [ ]:
df_pre_final = df_k_core.join(df.select("user_id", "parent_asin", "timestamp"),
                          on=["user_id", "parent_asin"], how="inner")

In [ ]:
df_pre_final.count(), df_k_core.count()

(939849, 857505)

кол-во строк в финальной версии больше, чем в целом было взаимодействий user&item (само собой, при общем кол-ве интеракций > k), скорее всего, иногда пользователи оставляют несколько отзывов на одни и те же айтемы (дубликаты по парам (user_id и parent_asin)), проверим датасет на наличие таких дубликатов

In [ ]:
dupes = df.groupBy("user_id", "parent_asin").agg(F.count("*").alias("cnt")) \
          .filter(F.col("cnt") > 1)
dupes.count()

57967

дубликаты реально обнаружены, оставим только самые ранние отзывы (т.к. юзер не может несколько раз купить одну и ту же игру, поэтому нам нет смысла несколько раз помещать один и тот же айтем в последовательность при предсказании)

In [ ]:
w = Window.partitionBy("user_id", "parent_asin").orderBy("timestamp")
df_final = df_pre_final.select("*", F.row_number().over(w).alias("seen_order")
).filter(F.col("seen_order") == 1).drop("seen_order")

In [ ]:
print(df_final.count())

829782


In [ ]:
df_final.write.mode('overwrite').parquet(f'{data_dir}/kcore_final')

## sequences

группируем по юзеру и составляем все его интеракции в виде списка

In [ ]:
seq1 = df_final.groupBy('user_id').agg(F.collect_list(F.struct('timestamp', 'parent_asin')).alias('interactions'))

сортируем список по timestamp (т.к. timestamp идет раньше parent_asin - не надо указывать, по чему сортируем) и удаляем неотсортированную версию

In [ ]:
seq2 = seq1.withColumn('interactions_sorted', F.sort_array('interactions')).drop('interactions')

убираем timestamp из данных, оставляем только сами айтемы

In [ ]:
seq3 = seq2.withColumn("item_seq", F.expr("transform(interactions_sorted, x -> x.parent_asin)")) \
           .drop("interactions_sorted")

добавим в данные длины последовательностей

In [ ]:
seq4 = seq3.withColumn('seq_len', F.size('item_seq'))

In [ ]:
seq4.select("seq_len").describe().show()

+-------+-----------------+
|summary|          seq_len|
+-------+-----------------+
|  count|            98906|
|   mean|8.389602248599681|
| stddev|7.829261555488236|
|    min|                1|
|    max|              474|
+-------+-----------------+



даже после k-core с k=5 имеем юзеров, с 1 айтемом в последовательности, это произошло из-за удаления дубликатов, т.к. юзер мог несколько раз писать отзыв на одну и ту же игру

избавимся от них

ограничение: из-за дедупликации после k-core свойство 5-core нарушено и для части товаров (у них могло остаться < 5 взаимодействий); корректнее было бы сначала удалить дубликаты, а потом делать k-core

In [ ]:
seq5 = seq4.filter(F.col("seq_len") >= 5)
seq5.select("seq_len").describe().show()

+-------+-----------------+
|summary|          seq_len|
+-------+-----------------+
|  count|            95268|
|   mean|8.596296762816475|
| stddev|7.901756428996507|
|    min|                5|
|    max|              474|
+-------+-----------------+



айдишники айтемов нужно сначала перевести из строк в числа

In [ ]:
uniq_items = df_final.select('parent_asin').distinct()
items_to_idx = uniq_items.rdd.map(lambda x: x['parent_asin']).zipWithIndex().collectAsMap()

In [ ]:
with open(f'{data_dir}/items_to_idx.json', 'w') as f:
    json.dump(items_to_idx, f)

In [ ]:
item_to_idx_broadcast = spark.sparkContext.broadcast(items_to_idx)
def map_items_to_idx(items):
    return [item_to_idx_broadcast.value[i] for i in items]

map_items = udf(map_items_to_idx, ArrayType(IntegerType()))
seq6 = seq5.withColumn("item_seq_idx", map_items(F.col("item_seq")))

leave-one-out split: `train_hist` — всё, кроме двух последних товаров, `val` — предпоследний, `test` — последний, `test_hist` — всё, кроме последнего

In [ ]:
seq7 = seq6.withColumn('train_hist', F.expr('slice(item_seq_idx, 1, size(item_seq_idx) - 2)'))
seq8 = seq7.withColumn('val', F.expr('element_at(item_seq_idx, -2)'))
seq9 = seq8.withColumn('test', F.expr('element_at(item_seq_idx, -1)'))
seq10 = seq9.withColumn('test_hist', F.expr('slice(item_seq_idx, 1, size(item_seq_idx) - 1)'))

In [ ]:
seq10.show(3)

+--------------------+--------------------+-------+--------------------+--------------------+-----+-----+--------------------+
|             user_id|            item_seq|seq_len|        item_seq_idx|          train_hist|  val| test|           test_hist|
+--------------------+--------------------+-------+--------------------+--------------------+-----+-----+--------------------+
|AE2252DKW4XJIZP5Q...|[B0050SX4CI, B002...|     11|[16045, 2967, 107...|[16045, 2967, 107...|10554|21092|[16045, 2967, 107...|
|AE22EOGVXOMYJ2QFJ...|[B008GEH9HO, B00C...|      6|[22999, 25270, 23...|[22999, 25270, 23...|15609| 8219|[22999, 25270, 23...|
|AE22HFOIMS3W4OYH5...|[B077XDQD2D, B078...|      6|[11356, 9298, 309...|[11356, 9298, 309...|15206|18248|[11356, 9298, 309...|
+--------------------+--------------------+-------+--------------------+--------------------+-----+-----+--------------------+
only showing top 3 rows


In [ ]:
seq10.write.mode("overwrite").parquet(f"{data_dir}/sequences_final")

## embeddings

In [ ]:
meta_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    filename="raw/meta_categories/meta_Video_Games.jsonl",
    repo_type="dataset"
)
df_meta_raw = spark.read.text(meta_path)

In [ ]:
df_meta_1 =df_meta_raw.select(
    F.get_json_object("value", "$.parent_asin").alias("parent_asin"),
    F.get_json_object("value", "$.title").alias("title"),
    F.from_json(F.get_json_object("value", "$.description"), ArrayType(StringType())).alias("description"),
    F.from_json(F.get_json_object("value", "$.features"), ArrayType(StringType())).alias("features"),)
df_meta_2 = df_meta_1.select('parent_asin',
                           F.concat_ws(
                              ' ',
                              F.col('title'),
                              F.array_join('features', ' '),
                              F.array_join('description', ' ')
                          ).alias('full_text'))

In [ ]:
df_meta_2.select(F.length('full_text')).describe().show()

+-------+------------------+
|summary| length(full_text)|
+-------+------------------+
|  count|            137269|
|   mean|1069.5066256765913|
| stddev|1330.6829279814922|
|    min|                 2|
|    max|             93999|
+-------+------------------+



In [ ]:
df_meta_2.filter(F.length('full_text') < 20).show(20, truncate=False)

+-----------+-------------------+
|parent_asin|full_text          |
+-----------+-------------------+
|B01N9HGQTM |EEEKit             |
|B08BJZNBYL |Keyboard 104 Keys  |
|B00851S9JU |Curling 2012       |
|B002EENE08 |Neo Fit Case       |
|B0032DTRBA |Red Storm Rising   |
|B089SPGH8K |EEEKit 11413       |
|B003QXMY48 |YAKUZA 4 (PS3)     |
|B08FYD7VNC |CURMIO Bag         |
|B00BSENXEC |PC CRYSIS 3 LTD    |
|B000SO6408 |A Vampyre Story    |
|B000ENMSXU |D-Day (PC) (UK)    |
|B003A02YL0 |Super Stocks       |
|B017SJBF3G |Xbox 360 Stand     |
|B089PWW14L |Grip               |
|B00AUXRCPI |for 9999 in 1      |
|B00005RUMH |AirXonix           |
|B0079DJ17Y |sniper elite v2    |
|B083W7DJ24 |Linkstyle 21232    |
|B0007PM0IS |Bioforge           |
|B07P8KT6DD |2                  |
+-----------+-------------------+
only showing top 20 rows


короткие описания в основном осмысленные, не сносим их и оставляем в виде контентных признаков

стоит обрезать лишнюю часть текста, т.к. эмбеддинг все равно получаем из первых n токенов (скорее всего 256), при этом, в full text в начале идет title + features (максимальная информативность), а далее description, где как раз может быть много бесполезной информации, но при этом также могут встретиться какие-то уникальные слова/описания, поэтому оставляем возможность посмотреть на него

использовать будем all-MiniLM-L6-v2, ее контекстное окно - 256 токенов, стоит оставлять ~1000 символов из каждого документа

In [ ]:
df_meta_3 = df_meta_2.withColumn('full_text_trunc',
                                 F.substring('full_text', 1, 1000))

In [ ]:
pdf_meta = df_meta_3.select("parent_asin", "full_text_trunc").toPandas()

In [ ]:
pdf_meta.shape

(137269, 2)

эмбеддинги считаем только для товаров из `items_to_idx` и строго в порядке их номеров: строка `i` матрицы — товар с номером `i`

In [ ]:
with open(f'{data_dir}/items_to_idx.json') as f:
    items_to_idx = json.load(f)
idx_to_items = {v: k for k, v in items_to_idx.items()}

pdf_items = pdf_meta.drop_duplicates('parent_asin').set_index('parent_asin')
missing = [a for a in items_to_idx if a not in pdf_items.index]
print('товаров:', len(items_to_idx), '| без меты:', len(missing))

texts = [pdf_items.loc[idx_to_items[i], 'full_text_trunc'] if idx_to_items[i] in pdf_items.index else ''
         for i in range(len(idx_to_items))]

st_model = SentenceTransformer("all-MiniLM-L6-v2")
item_embeddings = st_model.encode(texts, show_progress_bar=True, convert_to_numpy=True, batch_size=256)
print(item_embeddings.shape)

товаров: 26354 | без меты: 0


(26354, 384)


In [ ]:
np.save(f'{data_dir}/item_embeddings.npy', item_embeddings)